# Module 6 — Validation-Tuned Final Ensemble and Error Analysis

This notebook performs an explicit validation-only grid search over ensemble weights in 5% steps. Every component receives at least 5% weight. The selected weights are saved to `models/ensemble_weights.json`, then the held-out test set is evaluated exactly once with the selected configuration.

The deployed ensemble combines TF-IDF, Word2Vec, BiLSTM, and Transformer class probabilities. The existing conservative contrast rule is included during weight selection and final prediction.


In [1]:
from pathlib import Path
import sys
import re
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CLASS_NAMES, MODEL_DIR, RESULTS_DIR
from src.dataset_utils import load_fixed_data_splits
from src.evaluation import calculate_metrics, save_evaluation_outputs, update_metrics_file
from src.prediction import MODEL_ORDER, get_ensemble_predictor


_, validation_data, test_data = load_fixed_data_splits()
predictor = get_ensemble_predictor()

print("Loaded model order:", MODEL_ORDER)
print("Validation samples:", len(validation_data))
print("Test samples:", len(test_data))


Loaded model order: ('tfidf', 'word2vec', 'bilstm', 'transformer')
Validation samples: 2973
Test samples: 2974


## 1. Compute validation probabilities once

The four trained component models are run once on the validation split. Their probability matrices are reused for every candidate weight combination, so the grid search does not retrain or repeatedly rerun the models.


In [2]:
validation_texts = validation_data["original_text"].tolist()

processed_validation_texts, validation_components = (
    predictor.predict_component_probabilities(validation_texts)
)

validation_stack = np.stack(validation_components, axis=0)

validation_contrast_replacements = predictor.prepare_contrast_replacements(
    processed_validation_texts
)

print("Validation probability tensor shape:", validation_stack.shape)
print(
    "Contrast-rule replacement candidates:",
    len(validation_contrast_replacements),
)


Validation probability tensor shape: (4, 2973, 4)
Contrast-rule replacement candidates: 2


## 2. Search ensemble weights using validation Macro F1

Weights are searched in 5% increments. Each model must receive at least 5%, so a single component can receive at most 85%. The test set is not used anywhere in this selection step.

For every candidate, the same final contrast rule used in deployment is applied before calculating validation metrics.


In [3]:
STEP = 0.05
TOTAL_UNITS = int(round(1.0 / STEP))
MIN_UNITS = 1

true_validation_labels = validation_data["label"].tolist()
weight_search_results = []

for tfidf_units in range(MIN_UNITS, TOTAL_UNITS + 1):
    for word2vec_units in range(
        MIN_UNITS,
        TOTAL_UNITS - tfidf_units + 1,
    ):
        for bilstm_units in range(
            MIN_UNITS,
            TOTAL_UNITS - tfidf_units - word2vec_units + 1,
        ):
            transformer_units = (
                TOTAL_UNITS
                - tfidf_units
                - word2vec_units
                - bilstm_units
            )

            if transformer_units < MIN_UNITS:
                continue

            weights = np.array(
                [
                    tfidf_units,
                    word2vec_units,
                    bilstm_units,
                    transformer_units,
                ],
                dtype=np.float64,
            ) / TOTAL_UNITS

            combined_probabilities = predictor.combine_component_probabilities(
                validation_components,
                weights=weights,
                contrast_replacements=validation_contrast_replacements,
            )

            predicted_indices = combined_probabilities.argmax(axis=1)
            validation_predictions = [
                CLASS_NAMES[index]
                for index in predicted_indices
            ]

            metrics = calculate_metrics(
                true_validation_labels,
                validation_predictions,
            )

            weight_search_results.append(
                {
                    "TF-IDF Weight": weights[0],
                    "Word2Vec Weight": weights[1],
                    "BiLSTM Weight": weights[2],
                    "Transformer Weight": weights[3],
                    "Accuracy": metrics["Accuracy"],
                    "Macro Precision": metrics["Macro Precision"],
                    "Macro Recall": metrics["Macro Recall"],
                    "Macro F1": metrics["Macro F1"],
                }
            )

weight_search_df = pd.DataFrame(weight_search_results)

weight_search_df = weight_search_df.sort_values(
    by=[
        "Macro F1",
        "Accuracy",
        "Macro Recall",
        "Macro Precision",
    ],
    ascending=False,
).reset_index(drop=True)

print("Total weight combinations tested:", len(weight_search_df))
display(weight_search_df.head(20))


Total weight combinations tested: 969


,TF-IDF Weight,Word2Vec Weight,BiLSTM Weight,Transformer Weight,Accuracy,Macro Precision,Macro Recall,Macro F1
0,0.80,0.05,0.10,0.05,0.718803,0.691543,0.688129,0.689604
1,0.75,0.10,0.10,0.05,0.718466,0.690288,0.687182,0.688488
2,0.80,0.10,0.05,0.05,0.718466,0.688803,0.688312,0.688274
3,0.70,0.10,0.15,0.05,0.718130,0.691003,0.685523,0.687994
4,0.75,0.05,0.10,0.10,0.716784,0.691582,0.684378,0.687537
5,0.60,0.10,0.20,0.10,0.716784,0.694112,0.682276,0.687477
6,0.60,0.05,0.20,0.15,0.715775,0.696428,0.680586,0.687404
7,0.40,0.10,0.20,0.30,0.713757,0.702093,0.676630,0.686814
8,0.75,0.15,0.05,0.05,0.717457,0.686627,0.686942,0.686499
9,0.85,0.05,0.05,0.05,0.716112,0.686446,0.686582,0.686297


## 3. Select and save the best validation weights

The highest validation Macro F1 is the primary selection criterion. Accuracy, Macro Recall, and Macro Precision are deterministic tie-breakers. The full search table is saved for reporting and viva evidence.


In [4]:
best_result = weight_search_df.iloc[0]

best_weights = np.array(
    [
        best_result["TF-IDF Weight"],
        best_result["Word2Vec Weight"],
        best_result["BiLSTM Weight"],
        best_result["Transformer Weight"],
    ],
    dtype=np.float64,
)

print("BEST VALIDATION WEIGHTS")
print("-----------------------")
print(f"TF-IDF      : {best_weights[0]:.2f}")
print(f"Word2Vec    : {best_weights[1]:.2f}")
print(f"BiLSTM      : {best_weights[2]:.2f}")
print(f"Transformer : {best_weights[3]:.2f}")
print()
print(f"Validation Macro F1 : {best_result['Macro F1']:.4f}")
print(f"Validation Accuracy : {best_result['Accuracy']:.4f}")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

weight_search_path = RESULTS_DIR / "ensemble_weight_search.csv"
weight_search_df.to_csv(weight_search_path, index=False)

ensemble_weights_path = MODEL_DIR / "ensemble_weights.json"

weight_information = {
    "model_order": list(MODEL_ORDER),
    "weights": [float(weight) for weight in best_weights],
    "weights_by_model": {
        model_name: float(weight)
        for model_name, weight in zip(MODEL_ORDER, best_weights)
    },
    "selection_metric": "Validation Macro F1",
    "validation_macro_f1": float(best_result["Macro F1"]),
    "validation_accuracy": float(best_result["Accuracy"]),
    "validation_macro_precision": float(best_result["Macro Precision"]),
    "validation_macro_recall": float(best_result["Macro Recall"]),
    "search_step": STEP,
    "minimum_component_weight": STEP,
    "test_set_used_for_weight_selection": False,
}

with ensemble_weights_path.open("w", encoding="utf-8") as file:
    json.dump(
        weight_information,
        file,
        indent=4,
    )

# The predictor was already created before the JSON file was written,
# so update the current in-memory instance immediately.
predictor.component_weights = best_weights.copy()

print()
print("Saved full search table to:", weight_search_path)
print("Saved selected weights to:", ensemble_weights_path)


BEST VALIDATION WEIGHTS
-----------------------
TF-IDF      : 0.80
Word2Vec    : 0.05
BiLSTM      : 0.10
Transformer : 0.05

Validation Macro F1 : 0.6896
Validation Accuracy : 0.7188

Saved full search table to: E:\KUET\4-1\NLP Lab\Project\results\ensemble_weight_search.csv
Saved selected weights to: E:\KUET\4-1\NLP Lab\Project\models\ensemble_weights.json


## 4. Final validation summary and held-out test evaluation

After the weights are fixed, the test split is used for the final evaluation. No test result is used to change the selected weights.


In [5]:
validation_probabilities = predictor.combine_component_probabilities(
    validation_components,
    weights=best_weights,
    contrast_replacements=validation_contrast_replacements,
)

validation_predictions = [
    CLASS_NAMES[index]
    for index in validation_probabilities.argmax(axis=1)
]

validation_metrics = calculate_metrics(
    validation_data["label"],
    validation_predictions,
)

print("FINAL VALIDATION PERFORMANCE")
print("----------------------------")
for metric_name, metric_value in validation_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")


# The test set is touched only after validation-based weight selection is complete.
test_probabilities = predictor.predict_probabilities(
    test_data["original_text"].tolist()
)

test_predictions = [
    CLASS_NAMES[index]
    for index in test_probabilities.argmax(axis=1)
]

result = save_evaluation_outputs(
    experiment_id="M6.1",
    experiment_name="Validation-Tuned NLP Ensemble",
    family="Combined NLP System",
    test_data=test_data,
    predictions=test_predictions,
)

result["Representation"] = (
    "Word + Character TF-IDF + Word2Vec + BiLSTM + Transformer"
)
result["Validation Macro F1"] = validation_metrics["Macro F1"]
result["TF-IDF Weight"] = best_weights[0]
result["Word2Vec Weight"] = best_weights[1]
result["BiLSTM Weight"] = best_weights[2]
result["Transformer Weight"] = best_weights[3]

selected_metrics = update_metrics_file(
    pd.DataFrame([result])
)

display(selected_metrics)


FINAL VALIDATION PERFORMANCE
----------------------------
Accuracy: 0.7188
Macro Precision: 0.6915
Macro Recall: 0.6881
Macro F1: 0.6896


,Experiment ID,Model,Family,Accuracy,Macro Precision,Macro Recall,Macro F1,Representation,Validation Macro F1,TF-IDF Weight,Word2Vec Weight,BiLSTM Weight,Transformer Weight
0,M2.4,TF-IDF + Logistic Regression,Discriminative,0.739072,0.745713,0.696072,0.713898,Word + Character TF-IDF,0.706673,NaN,NaN,NaN,NaN
1,M3.2,TF-IDF Weighted Word2Vec + Logistic Regression,Dense Embedding,0.644250,0.609199,0.631593,0.615919,TF-IDF Weighted Word2Vec,0.617307,NaN,NaN,NaN,NaN
2,M4.2,Bidirectional LSTM,Sequence Neural,0.682582,0.679693,0.652137,0.662581,Trainable Embedding + BiLSTM,0.649343,NaN,NaN,NaN,NaN
3,M5.1,Transformer Encoder,Self-Attention,0.687962,0.694336,0.648622,0.664546,Embedding + Positional Encoding,0.637101,NaN,NaN,NaN,NaN
4,M6.1,Validation-Tuned NLP Ensemble,Combined NLP System,0.725622,0.699235,0.701875,0.700386,Word + Character TF-IDF + Word2Vec + BiLSTM + ...,0.689604,0.8,0.05,0.1,0.05


## 5. Categorize real ensemble errors

The error-analysis logic is retained from the original notebook and now uses predictions from the validation-selected final ensemble.


In [6]:
negation_pattern = re.compile(
    r"\b(?:not|no|never|na|nai|nei|nay|nahi|dont|doesnt|didnt|isnt|cant|wont)\b"
)
contrast_pattern = re.compile(r"\b(?:but|however|kintu|tobe)\b")
spelling_variants = {
    "bhalo",
    "valo",
    "vhalo",
    "vaalo",
    "kharap",
    "kharappp",
}


def categorize_error(row):
    text = str(row["processed_text"]).lower()
    tokens = text.split()

    if len(tokens) <= 3:
        return "Very short text"
    if negation_pattern.search(text):
        return "Negation"
    if contrast_pattern.search(text) or row["label"] == "Mixed":
        return "Mixed or contrastive sentiment"
    if spelling_variants.intersection(tokens):
        return "Romanized spelling variation"

    return "Other / manual review needed"


error_analysis = test_data[
    [
        "sample_id",
        "original_text",
        "processed_text",
        "label",
    ]
].copy()

error_analysis["predicted_label"] = test_predictions
error_analysis["confidence"] = test_probabilities.max(axis=1)

error_analysis = error_analysis[
    error_analysis["label"] != error_analysis["predicted_label"]
].copy()

error_analysis["error_category"] = error_analysis.apply(
    categorize_error,
    axis=1,
)

error_analysis_path = RESULTS_DIR / "error_analysis.csv"
error_analysis.to_csv(
    error_analysis_path,
    index=False,
)

print("Saved error analysis to:", error_analysis_path)
display(error_analysis["error_category"].value_counts())
display(error_analysis.head(20))


Saved error analysis to: E:\KUET\4-1\NLP Lab\Project\results\error_analysis.csv


error_category
Other / manual review needed      453
Negation                          184
Mixed or contrastive sentiment    109
Romanized spelling variation       64
Very short text                     6
Name: count, dtype: int64

,sample_id,original_text,processed_text,label,predicted_label,confidence,error_category
2,16,Walton din din khub valo korteche. Ar xiaomi d...,walton din din khub valo korteche ar xiaomi di...,Mixed,Positive,0.540459,Mixed or contrastive sentiment
6,35,Video ta gorom er shurute deya uchit cilo apne...,video ta gorom er shurute deya uchit cilo apne...,Mixed,Neutral,0.382264,Mixed or contrastive sentiment
7,37,Video guli kharaf na. Tobe kotha bolar dhoron ...,video guli kharaf na tobe kotha bolar dhoron m...,Mixed,Negative,0.372752,Negation
10,53,Valo....khub valo na....cole,valo khub valo na cole,Mixed,Positive,0.579133,Negation
16,119,Vaiya hsc chemistry math physics er playlists ...,vaiya hsc chemistry math physics er playlists ...,Mixed,Positive,0.358871,Negation
19,130,Vai phone kinar pore egula vdo dhekle kmn lage,vai phone kinar pore egula vdo dhekle kmn lage,Mixed,Negative,0.341487,Mixed or contrastive sentiment
20,143,"Vai apnar review valo lage tai dekhtam ,but ma...",vai apnar review valo lage tai dekhtam but maj...,Mixed,Positive,0.541925,Mixed or contrastive sentiment
21,146,Vai apar sound ar vitor noise ase onk asa kori...,vai apar sound ar vitor noise ase onk asa kori...,Mixed,Positive,0.348427,Mixed or contrastive sentiment
23,152,Vai aj gorib bole ios calai borolok hoile xiao...,vai aj gorib bole ios calai borolok hoile xiao...,Mixed,Negative,0.428870,Mixed or contrastive sentiment
24,154,"Vai , apnader content onk valo hoileo sound qu...",vai apnader content onk valo hoileo sound qual...,Mixed,Negative,0.481255,Negation
